# K-NumASR: Korean Numeric Speech Recognition Benchmark

Colab에서 `런타임 > 런타임 유형 변경 > T4 GPU`로 설정한 뒤 `런타임 > 모두 실행`을 누르세요. 마지막 셀이 데이터셋, 예측 결과, 리더보드를 zip으로 내려받습니다.

In [ ]:
!nvidia-smi -L

In [ ]:
REPO_URL = "https://github.com/JinsooKim0118/K-NumASR.git"
!git clone -q $REPO_URL /content/K-NumASR
%cd /content/K-NumASR
!ls

In [ ]:
!pip install -q -r requirements.txt

## 1. 평가 데이터셋 구축
합성 숫자 발화 세트(5개 유형 × 60문장)를 TTS로 만들고, FLEURS-ko에서 숫자가 포함된 발화를 추출합니다.

In [ ]:
!python scripts/build_synthetic_text.py --out data/synthetic/metadata.csv --per_category 60
!python scripts/synthesize_tts.py --meta data/synthetic/metadata.csv --backend edge

In [ ]:
!python scripts/build_fleurs_subset.py --out data/fleurs_num --splits dev test

In [ ]:
import pandas as pd
syn = pd.read_csv('data/synthetic/metadata.csv')
fl = pd.read_csv('data/fleurs_num/metadata.csv')
print(syn.category.value_counts())
print('FLEURS numeric:', len(fl))
syn.head()

In [ ]:
from IPython.display import Audio, display
for _, r in syn.sample(3, random_state=0).iterrows():
    print(r.category, '|', r.text)
    display(Audio('data/synthetic/' + r.audio))

## 2. Baseline 및 제안 방법 추론

In [ ]:
SYSTEMS = [
    ("openai/whisper-tiny", "whisper-tiny", "whisper", False),
    ("openai/whisper-base", "whisper-base", "whisper", False),
    ("openai/whisper-small", "whisper-small", "whisper", False),
    ("openai/whisper-medium", "whisper-medium", "whisper", False),
    ("openai/whisper-large-v3-turbo", "whisper-large-v3-turbo", "whisper", False),
    ("openai/whisper-large-v3", "whisper-large-v3", "whisper", False),
    ("openai/whisper-small", "whisper-small+NumPrompt", "whisper", True),
    ("openai/whisper-large-v3-turbo", "whisper-large-v3-turbo+NumPrompt", "whisper", True),
    ("kresnik/wav2vec2-large-xlsr-korean", "wav2vec2-xlsr-ko", "ctc", False),
]

In [ ]:
import os, subprocess
for model, name, typ, prompt in SYSTEMS:
    if os.path.exists(f'predictions/{name}.jsonl'):
        print('skip', name)
        continue
    cmd = ['python', 'scripts/run_asr.py', '--model', model, '--system', name, '--type', typ]
    if prompt:
        cmd.append('--prompt')
    print('>>', ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-500:])
    if r.returncode != 0:
        print('FAILED', name, r.stderr[-2000:])

## 3. 평가 및 리더보드

In [ ]:
!python scripts/evaluate.py --pred_dir predictions --out_dir results

In [ ]:
import pandas as pd
pd.read_csv('results/leaderboard.csv')

## 4. 결과 내려받기
`knumasr_results.zip`을 내려받아 로컬 `K-NumASR` 폴더에 압축을 풀면 됩니다.

In [ ]:
!zip -qr knumasr_results.zip data predictions results
from google.colab import files
files.download('knumasr_results.zip')